# RL upper-bound improver for unknotting numbers

This notebook is designed to live **inside a GitHub-style repository** and run
**without Google Drive**.

Expected local layout:

```text
repo-root/
├─ upper_bound_unknotting_v6_local.ipynb
├─ requirements.txt
├─ README.md
├─ data/
│  └─ unknotting.xlsx
├─ models/
│  └─ best_model.zip                 # optional
├─ training_data/                    # optional
│  ├─ hard_unknots.csv
│  ├─ very_hard_unknots.csv
│  └─ random_diagrams.csv
└─ outputs/
```

The notebook does the following:

1. loads `data/unknotting.xlsx`
2. fills missing Jones vectors from the PD presentation when needed
3. finds knots whose unknotting number is given by a range such as `[2,3]`
4. inflates the diagram
5. flips one crossing at a time
6. runs the RL unknotter / reducer
7. computes the Jones vector of the reduced knot
8. identifies all matching knots in the database, allowing mirrors
9. takes the **maximum** matching upper bound
10. improves the original upper bound when possible by setting it to `[old_lower, 1 + matched_upper_bound]`
11. saves the workbook after each processed knot

If no pretrained model is present, the notebook will try to train one from
local training files. If those are absent too, it falls back to PD data already
present in `unknotting.xlsx`, so the repository can still run in a self-contained
way.

The Jones-vector convention used here is:

`[lowest power of q, highest power of q, a, b, c, ...]`

with all intermediate coefficients included, including zeros.


In [ ]:
# Local setup (run once per environment, if needed)
# %pip install -r requirements.txt



In [ ]:
import os, re, json, ast, math, random, csv, glob, time
from pathlib import Path
from dataclasses import dataclass
from fractions import Fraction
from collections import defaultdict
from typing import Iterable, Optional, List, Tuple

import numpy as np
import pandas as pd

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

from tqdm.auto import tqdm

import snappy
from spherogram import Link

SEED = 42
random.seed(SEED)
np.random.seed(SEED)



In [ ]:
# Local repository paths (no Google Drive needed)
from pathlib import Path
import sys

_candidate_roots = [Path.cwd(), Path.cwd().parent]
_repo_root = next(
    (candidate for candidate in _candidate_roots if (candidate / "src").exists()),
    Path.cwd(),
)
_src_dir = _repo_root / "src"
if _src_dir.exists() and str(_src_dir) not in sys.path:
    sys.path.insert(0, str(_src_dir))

from upperbounds.io.paths import prepare_notebook_paths

PATH_CONTEXT = prepare_notebook_paths(repo_root=_repo_root)
globals().update(PATH_CONTEXT.as_notebook_globals())
PATH_CONTEXT.print_summary()


In [ ]:
# ----------------------------
# Configuration
# ----------------------------
# Which unresolved knots to process:
#   PROCESS_MODE = "all"              -> all range-valued unknotting entries, i.e. all [a,b] with a != b
#   PROCESS_MODE = "first_n"          -> first N range-valued entries
#   PROCESS_MODE = "slice"            -> entries [START_INDEX:END_INDEX] among range-valued entries
#   PROCESS_MODE = "bounds_eq"        -> only entries with unknotting number exactly [TARGET_LOWER, TARGET_UPPER]
#   PROCESS_MODE = "bounds_eq_slice"  -> first filter to [TARGET_LOWER, TARGET_UPPER], then take [START_INDEX:END_INDEX]
#   PROCESS_MODE = "bounds_neq"       -> explicitly all [a,b] with a != b (same target set as "all")
#   PROCESS_MODE = "bounds_neq_slice" -> explicitly all [a,b] with a != b, then take [START_INDEX:END_INDEX]
#   PROCESS_MODE = "crossing_number"   -> all knots with TARGET_CROSSING_NUMBER, sorted by knot number
PROCESS_MODE = "crossing_number"

TARGET_LOWER = 1
TARGET_UPPER = 14
TARGET_CROSSING_NUMBER = 14

START_INDEX = 0          # 0-based within the filtered target list
END_INDEX = None          # exclusive; None means "through the end"
FIRST_N = 100              # used only when PROCESS_MODE == "first_n"

# Keep original search-quality defaults.
NUM_VARIANTS_PER_KNOT = 12
BACKTRACK_STEPS_MIN = 6
BACKTRACK_STEPS_MAX = 8
RIII_STEPS_MAX = 20
UNKNOTTER_EPISODES_PER_FLIP = 1
UNKNOTTER_MAX_STEPS = 500

TRAIN_IF_MODEL_MISSING = True
TRAIN_STEPS_IF_NEEDED = 20000

# Saving / output
SAVE_AFTER_EACH_TARGET = True
SAVE_EVERY_KNOT = True          # robust alias used in the main loop
WRITE_UPDATED_COPY = False
MAKE_TIMESTAMPED_BACKUP = False
# No backup file and no secondary updated copy: only overwrite XLSX_PATH

# Only trust Jones identification when the reduced PD lies in the database range.
MIN_DATABASE_CROSSINGS = 1
MAX_DATABASE_CROSSINGS = 13

# Optional safety valve for very long runs:
MAX_FLIPS_PER_VARIANT = None   # set to an integer to cap crossing flips per inflated variant

MODEL_PATH_CANDIDATES = [
    BASE / "best_model.zip",
    BASE / "ppo_knot_rl_spherogram_continued.zip",
    OUT_DIR / "best_model.zip",
]

# Training data sources from the original notebook / paper pipeline
GCS_CSV_PATH_MAIN = "gs://gdm-unknotting/hard_unknots.csv"
GCS_CSV_PATH_VERY = "gs://gdm-unknotting/very_hard_unknots.csv"

# Optional local extras: only used if present
LOCAL_EXTRA_FILES = [
    BASE / "random_diagrams.csv",
    BASE / "random_diagrams.txt",
    BASE / "hard_unknots.csv",
    BASE / "very_hard_unknots.csv",
]


# ----------------------------
# Parallelization settings
# ----------------------------
ENABLE_PARALLEL = False
PARALLEL_MAX_WORKERS = None      # kept for compatibility with older cells
PARALLEL_RESERVE_CORES = 1       # keep some CPU cores free for OS / notebook responsiveness
PARALLEL_INFERENCE_DEVICE = "cpu"



In [ ]:
# ----------------------------
# Helpers: workbook / parsing
# ----------------------------
import sys
from pathlib import Path

try:
    _repo_root = REPO_ROOT
except NameError:
    _repo_root = Path.cwd()
_src_dir = _repo_root / "src"
if _src_dir.exists() and str(_src_dir) not in sys.path:
    sys.path.insert(0, str(_src_dir))

from upperbounds.data.parsing import (
    canon_coeff_key,
    canon_coeff_key_mirror,
    ensure_minmax_coeffs,
    format_unknotting,
    normalize_invariant_key_cell,
    parse_pd_cell,
    parse_unknotting_entry,
    parse_vector_cell,
    pick_first_existing,
    span_abs,
    strip_leading_trailing_zeros,
)


In [ ]:
# ----------------------------
# Load workbook and identify columns
# ----------------------------
from upperbounds.io.workbook import load_workbook_with_columns

df, WORKBOOK_COLUMNS = load_workbook_with_columns(XLSX_PATH)
globals().update(WORKBOOK_COLUMNS.as_notebook_globals())
WORKBOOK_COLUMNS.print_summary()

display(df.head())


In [ ]:
# ----------------------------
# Jones polynomial / invariant helpers
# ----------------------------
from upperbounds.invariants.jones import (
    bracket_from_pd,
    crossing_sign_pd,
    jones_string_from_pd,
    jones_vector_from_pd,
    parse_jones_string_to_dict,
    poly_add,
    poly_dict_to_knotinfo_vector,
    poly_monom,
    poly_mul,
    poly_scale,
)


In [ ]:
# ----------------------------
# Fill missing Jones vectors in the database itself if needed
# ----------------------------
missing_before = 0
filled = 0
errors = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Filling missing Jones vectors"):
    vec = parse_vector_cell(row.get(jones_col))
    if ensure_minmax_coeffs(vec) is not None:
        continue

    missing_before += 1
    pd_list = parse_pd_cell(row.get(pd_col))
    if pd_list is None:
        errors.append((idx, row.get(knot_col), "missing/bad PD"))
        continue

    if len(pd_list) > 20:
        # the direct bracket expansion is exponential in crossing number
        # so we skip very large PDs here; the improvement pipeline only
        # needs Jones vectors for rows that are actually matched later.
        continue

    new_vec, err = jones_vector_from_pd(pd_list)
    if new_vec is not None:
        df.at[idx, jones_col] = str(new_vec)
        filled += 1
    else:
        errors.append((idx, row.get(knot_col), err))

print("Missing Jones entries before:", missing_before)
print("Filled directly from PD:", filled)
print("Unfilled / errors:", len(errors))
if errors[:10]:
    print("Sample errors:", errors[:10])



In [ ]:
# ----------------------------
# Build Jones lookup from the workbook
# The requested rule is:
#   if multiple knots have the same Jones polynomial,
#   take the MAX over all matching upper bounds
# ----------------------------
lookup = defaultdict(list)

for idx, row in df.iterrows():
    vec = parse_vector_cell(row.get(jones_col))
    parsed = ensure_minmax_coeffs(vec)
    if parsed is None:
        continue

    uk = parse_unknotting_entry(row.get(u_col))
    if uk["upper"] is None:
        continue

    mn, mx, coeffs = parsed
    sp = span_abs(mn, mx)
    rec = {
        "row_index": int(idx),
        "knot": row.get(knot_col),
        "upper": int(uk["upper"]),
        "lower": None if uk["lower"] is None else int(uk["lower"]),
        "mirror": False,
    }
    lookup[(sp, canon_coeff_key(coeffs))].append({**rec, "mirror": False})
    lookup[(sp, canon_coeff_key_mirror(coeffs))].append({**rec, "mirror": True})

print("Lookup keys:", len(lookup))



In [ ]:
# ----------------------------
# Training / RL utilities
# ----------------------------
from upperbounds.rl.runtime import (
    EnvCfg,
    SphKnotEnv,
    clean_pd_lines,
    crossings,
    default_local_training_files,
    default_model_path_candidates,
    is_trivial_zero,
    load_or_train_ppo_model,
    load_training_pd_lines as _load_training_pd_lines,
    make_sb3_load_custom_objects,
    make_single_env,
    parse_link_strict,
    read_first_col_local,
    riii_shuffle_only_link,
    run_unknotter_on_pd,
    workbook_pd_lines as _workbook_pd_lines,
)

MODEL_PATH_CANDIDATES = default_model_path_candidates(MODELS_DIR, OUT_DIR)
LOCAL_EXTRA_FILES = default_local_training_files(TRAINING_DIR, DATA_DIR)


def workbook_pd_lines(max_keep: int | None = None) -> list[str]:
    return _workbook_pd_lines(df, pd_col, max_keep=max_keep)


def load_training_pd_lines() -> list[str]:
    return _load_training_pd_lines(
        local_extra_files=LOCAL_EXTRA_FILES,
        df=df,
        pd_col=pd_col,
        seed=SEED,
    )


In [ ]:
# ----------------------------
# Load or train PPO model
# ----------------------------
cfg = EnvCfg(max_steps=UNKNOTTER_MAX_STEPS, allow_backtrack=True, seed=SEED)

if "MODEL_PATH_CANDIDATES" not in globals():
    MODEL_PATH_CANDIDATES = default_model_path_candidates(MODELS_DIR, OUT_DIR)
if "LOCAL_EXTRA_FILES" not in globals():
    LOCAL_EXTRA_FILES = default_local_training_files(TRAINING_DIR, DATA_DIR)

model, best_model_path = load_or_train_ppo_model(
    model_path_candidates=MODEL_PATH_CANDIDATES,
    train_if_model_missing=TRAIN_IF_MODEL_MISSING,
    train_steps_if_needed=TRAIN_STEPS_IF_NEEDED,
    cfg=cfg,
    local_extra_files=LOCAL_EXTRA_FILES,
    df=df,
    pd_col=pd_col,
    seed=SEED,
    output_model_path=OUT_DIR / "best_model.zip",
)
ACTIVE_MODEL_PATH = Path(best_model_path)
print("Active model path:", ACTIVE_MODEL_PATH)


In [ ]:
# ----------------------------
# Inflation and flip helpers
# ----------------------------
def flip_crossing_quad(quad):
    a, b, c, d = quad
    return [b, c, d, a]

def generate_inflated_variants(pd_list,
                               num_variants=10,
                               backtrack_steps_min=6,
                               backtrack_steps_max=8,
                               riii_steps_max=20):
    variants = []
    L0 = Link(pd_list)

    for _ in range(num_variants):
        pd0 = L0.PD_code()
        if isinstance(pd0, list):
            L = Link(pd0)
        else:
            L = Link([[int(getattr(e, "label", e)) for e in vtx] for vtx in pd0])

        steps = random.randint(backtrack_steps_min, backtrack_steps_max)
        try:
            L.backtrack(steps=steps, prob_type_1=0.35, prob_type_2=0.65)
        except Exception:
            pass
        try:
            L, _ = riii_shuffle_only_link(L, min(riii_steps_max, steps))
        except Exception:
            pass

        try:
            pd_new = [list(q) for q in L.PD_code()]
            variants.append(pd_new)
        except Exception:
            variants.append([list(q) for q in pd_list])

    if not variants:
        variants = [[list(q) for q in pd_list]]
    return variants

def generate_single_flip_variants(pd_list):
    out = []
    n = len(pd_list)
    for i in range(n):
        flipped = []
        for j, quad in enumerate(pd_list):
            flipped.append(flip_crossing_quad(quad) if i == j else list(quad))
        out.append((i, flipped))
    return out

def match_jones_vector_to_database(vec):
    parsed = ensure_minmax_coeffs(vec)
    if parsed is None:
        return []
    mn, mx, coeffs = parsed
    sp = span_abs(mn, mx)
    return lookup.get((sp, canon_coeff_key(coeffs)), [])

def best_upper_bound_from_matches(matches):
    if not matches:
        return None
    uppers = [m["upper"] for m in matches if m.get("upper") is not None]
    if not uppers:
        return None
    return max(uppers)



In [ ]:
# ----------------------------
# Choose target rows
# ----------------------------
from upperbounds.pipeline.targets import (
    knot_crossing_and_number,
    print_target_summary,
    select_targets,
)

TARGET_SELECTION = select_targets(
    df=df,
    knot_col=knot_col,
    u_col=u_col,
    process_mode=PROCESS_MODE,
    target_lower=TARGET_LOWER,
    target_upper=TARGET_UPPER,
    start_index=START_INDEX,
    end_index=END_INDEX,
    first_n=FIRST_N if "FIRST_N" in globals() else 100,
    target_crossing_number=TARGET_CROSSING_NUMBER,
)
globals().update(TARGET_SELECTION.as_notebook_globals())
print_target_summary(
    TARGET_SELECTION,
    TARGET_LOWER,
    TARGET_UPPER,
    target_crossing_number=TARGET_CROSSING_NUMBER,
)

if len(targets) > 0:
    preview = pd.DataFrame(targets[:10])
    display(preview)
else:
    print("No targets selected.")


In [ ]:
# ----------------------------
# Main improvement loop
# ----------------------------
# Robust defaults in case cells were run out of order
if "SAVE_EVERY_KNOT" not in globals():
    SAVE_EVERY_KNOT = True
if "MIN_DATABASE_CROSSINGS" not in globals():
    MIN_DATABASE_CROSSINGS = 1
if "MAX_DATABASE_CROSSINGS" not in globals():
    MAX_DATABASE_CROSSINGS = 13

results_all = []
timestamp = time.strftime("%Y%m%d-%H%M%S")

for tnum, target in enumerate(targets, start=1):
    idx = target["row_index"]
    knot_name = target["knot"]
    current_lower = target["lower"]
    current_upper = target["upper"]

    print("\n" + "="*80)
    print(f"[{tnum}/{len(targets)}] Processing:", knot_name, " current range:", [current_lower, current_upper])

    original_pd = parse_pd_cell(df.at[idx, pd_col])
    if original_pd is None:
        print("  Skipping: bad or missing PD.")
        results_all.append({
            "row_index": idx,
            "knot": knot_name,
            "status": "bad_pd",
        })
        continue

    base_vec = parse_vector_cell(df.at[idx, jones_col])
    if ensure_minmax_coeffs(base_vec) is None and len(original_pd) <= 20:
        new_vec, err = jones_vector_from_pd(original_pd)
        if new_vec is not None:
            df.at[idx, jones_col] = str(new_vec)

    variants = generate_inflated_variants(
        original_pd,
        num_variants=NUM_VARIANTS_PER_KNOT,
        backtrack_steps_min=BACKTRACK_STEPS_MIN,
        backtrack_steps_max=BACKTRACK_STEPS_MAX,
        riii_steps_max=RIII_STEPS_MAX,
    )

    best_new_upper = current_upper
    best_evidence = None
    tried = 0
    matched = 0

    for vnum, variant_pd in enumerate(variants, start=1):
        flip_variants = generate_single_flip_variants(variant_pd)
        if MAX_FLIPS_PER_VARIANT is not None:
            flip_variants = flip_variants[:MAX_FLIPS_PER_VARIANT]

        for flip_index, flipped_pd in flip_variants:
            tried += 1
            success, min_crossings_found, best_pd = run_unknotter_on_pd(
                flipped_pd,
                model,
                cfg,
                episodes=UNKNOTTER_EPISODES_PER_FLIP,
                return_best_pd=True,
            )

            if best_pd is None:
                continue

            reduced_crossings = len(best_pd)

            # Only trust Jones matching inside the database crossing range.
            if not (MIN_DATABASE_CROSSINGS <= reduced_crossings <= MAX_DATABASE_CROSSINGS):
                continue

            vec, err = jones_vector_from_pd(best_pd)
            if vec is None:
                continue

            matches = match_jones_vector_to_database(vec)
            if not matches:
                continue

            matched += 1
            matched_upper = best_upper_bound_from_matches(matches)
            if matched_upper is None:
                continue

            candidate_upper = matched_upper + 1

            if candidate_upper < best_new_upper:
                best_new_upper = candidate_upper
                best_evidence = {
                    "variant_index": vnum,
                    "flip_index": int(flip_index),
                    "matched_upper": int(matched_upper),
                    "candidate_upper": int(candidate_upper),
                    "rl_success": bool(success),
                    "min_crossings_found": int(min_crossings_found),
                    "best_pd_crossings": reduced_crossings,
                    "matched_knots": sorted(
                        {(str(m["knot"]), int(m["upper"]), bool(m["mirror"])) for m in matches},
                        key=lambda x: (x[0], x[1], x[2])
                    ),
                    "best_pd": best_pd,
                    "jones_vector": vec,
                }
                print(f"  Improved via variant {vnum}, flip {flip_index}: upper {current_upper} -> {best_new_upper}")

    improved = best_new_upper < current_upper
    if improved:
        df.at[idx, u_col] = format_unknotting(current_lower, best_new_upper)

        if SAVE_EVERY_KNOT:
            df.to_excel(XLSX_PATH, index=False)
            print("  Saved updated workbook to:", XLSX_PATH)
    else:
        print("  No improvement found.")

    result = {
        "row_index": idx,
        "knot": knot_name,
        "old_lower": current_lower,
        "old_upper": current_upper,
        "new_upper": best_new_upper,
        "improved": improved,
        "tried_flip_runs": tried,
        "matched_flip_runs": matched,
        "evidence": best_evidence,
    }
    results_all.append(result)

    jsonl_path = OUT_DIR / "upper_bound_pass_results.jsonl"
    with open(jsonl_path, "a") as f:
        f.write(json.dumps(result) + "\n")

print("\nDone.")


12a_49